# Ce projet analyse les ventes hebdomadaires de 45 magasins Walmart pour identifier les facteurs qui influencent les performances et développer un modèle prédictif. Le dataset contient environ 6400 observations avec des variables économiques (CPI, chômage, prix du carburant), météorologiques (température) et un indicateur de jours fériés. L'objectif est de comprendre l'impact de ces facteurs sur les ventes et de créer un modèle de régression linéaire capable de prédire les ventes futures

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
from sklearn.linear_model import LinearRegression
import seaborn as sns 
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/walmart-sales/Walmart_Sales.csv


# Importation et Nettoyage des données 


In [2]:
#Importation du dataset
df = pd.read_csv("/kaggle/input/walmart-sales/Walmart_Sales.csv")
df

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,1,05-02-2010,1643690.90,0,42.31,2.572,211.096358,8.106
1,1,12-02-2010,1641957.44,1,38.51,2.548,211.242170,8.106
2,1,19-02-2010,1611968.17,0,39.93,2.514,211.289143,8.106
3,1,26-02-2010,1409727.59,0,46.63,2.561,211.319643,8.106
4,1,05-03-2010,1554806.68,0,46.50,2.625,211.350143,8.106
...,...,...,...,...,...,...,...,...
6430,45,28-09-2012,713173.95,0,64.88,3.997,192.013558,8.684
6431,45,05-10-2012,733455.07,0,64.89,3.985,192.170412,8.667
6432,45,12-10-2012,734464.36,0,54.47,4.000,192.327265,8.667
6433,45,19-10-2012,718125.53,0,56.47,3.969,192.330854,8.667


In [3]:
df.isna().sum()

Store           0
Date            0
Weekly_Sales    0
Holiday_Flag    0
Temperature     0
Fuel_Price      0
CPI             0
Unemployment    0
dtype: int64

In [4]:
df.duplicated().sum()

np.int64(0)

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6435 entries, 0 to 6434
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         6435 non-null   int64  
 1   Date          6435 non-null   object 
 2   Weekly_Sales  6435 non-null   float64
 3   Holiday_Flag  6435 non-null   int64  
 4   Temperature   6435 non-null   float64
 5   Fuel_Price    6435 non-null   float64
 6   CPI           6435 non-null   float64
 7   Unemployment  6435 non-null   float64
dtypes: float64(5), int64(2), object(1)
memory usage: 402.3+ KB


In [6]:
df.columns

Index(['Store', 'Date', 'Weekly_Sales', 'Holiday_Flag', 'Temperature',
       'Fuel_Price', 'CPI', 'Unemployment'],
      dtype='object')

In [7]:
df.describe()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,6435.000000,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
std,12.988182,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885
min,1.000000,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000


# Predictions des ventes 

In [8]:
df = df.drop(['Date'], axis=1)
df = pd.get_dummies(df, columns=['Store'], drop_first =True)
df

,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Store_2,Store_3,Store_4,Store_5,...,Store_36,Store_37,Store_38,Store_39,Store_40,Store_41,Store_42,Store_43,Store_44,Store_45
0,1643690.90,0,42.31,2.572,211.096358,8.106,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,1641957.44,1,38.51,2.548,211.242170,8.106,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,1611968.17,0,39.93,2.514,211.289143,8.106,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,1409727.59,0,46.63,2.561,211.319643,8.106,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,1554806.68,0,46.50,2.625,211.350143,8.106,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6430,713173.95,0,64.88,3.997,192.013558,8.684,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
6431,733455.07,0,64.89,3.985,192.170412,8.667,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
6432,734464.36,0,54.47,4.000,192.327265,8.667,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True
6433,718125.53,0,56.47,3.969,192.330854,8.667,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [9]:
X = df.drop("Weekly_Sales", axis=1). values
y = df[ "Weekly_Sales"]. values
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state=42)
reg = LinearRegression ()
reg. fit(X_train,y_train)
y_pred = reg.predict(X_test)
print(f"le score de prédiction est de {round(reg. score(X_test, y_test)*100)}%")
print(f"la prediction des ventes moyenne est ,{y_pred.mean().round(0)}$")
print(f"le rmse est de , {root_mean_squared_error(y_pred,y_test)}")
print(f"la moyenne des ventes actuelles,{y_test.mean().round(0)}$")

le score de prédiction est de 92%
la prediction des ventes moyenne est ,1062135.0$
le rmse est de , 163401.66817552678
la moyenne des ventes actuelles,1060167.0$


X = df.drop("Weekly_Sales", axis=1).values
y = df["Weekly_Sales"].values
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.3,random_state=42)
reg = LinearRegression()
reg.fit(X_train,y_train)
y_pred = reg.predict(X_test)
print(reg.score(X_test,y_test))
print(y_pred)
print(root_mean_squared_error(y_test,y_pred))
print(y_test)


In [10]:
coef_df = pd.DataFrame({"Names" : df.drop("Weekly_Sales", axis=1).columns, "Coef" : reg.coef_}).sort_values("Coef", ascending =False)
coef_df

,Names,Coef
7,Store_4,7.473911e+05
16,Store_13,6.927736e+05
13,Store_10,6.319772e+05
23,Store_20,5.772044e+05
17,Store_14,5.469690e+05
30,Store_27,4.771565e+05
5,Store_2,3.735837e+05
22,Store_19,1.388886e+05
31,Store_28,1.359570e+05
0,Holiday_Flag,7.575748e+04


# les magasins font plus de ventes pendant les jours fériés, lorsque le prix du fuel est moindre et qu'il fait beau. les stores qui vendent le mieux sont les stoeres n° 4, 13,10,20,14,27,2,19 et 28. 
